# 第 10 章 大语言模型与智能体（上）：从零实现 nanoGPT

**大语言模型**（Large Language Model，LLM）是 2023 年以来深度学习领域最显赫的成果。GPT、Llama、Claude、Qwen 等模型展现出来的**涌现能力**（上下文学习、思维链推理、零样本泛化等）已经彻底改变了 AI 的应用形态。

本节从最基础的语言建模任务出发，从零实现一个**字符级 nanoGPT**：约 200 行代码、~1M 参数，在 TinyShakespeare 上预训练几分钟就能生成像模像样的伪莎士比亚文本。

涵盖内容：

- **自回归语言建模**：next-token prediction 的训练目标；
- **字符级 / BPE 分词**：把原始文本切成 token；
- **Transformer 解码器**：因果注意力 + 多头注意力 + 前馈 + 残差；
- **预训练循环**：cross-entropy loss、AdamW、学习率 warmup + cosine decay；
- **采样策略**：greedy / temperature / top-k / top-p；
- **KV cache**：增量解码的工程优化。


## 1. 语言模型回顾：自回归与 next-token 预测

**语言模型**（Language Model）的任务是给定一段文本前缀 $x_1, x_2, \ldots, x_{t-1}$，预测下一个 token $x_t$ 的条件概率分布

$$ p(x_t \mid x_1, \ldots, x_{t-1}). $$

通过链式法则，可以把整段文本的概率写成

$$ p(x_1, x_2, \ldots, x_T) = \prod_{t=1}^T p(x_t \mid x_{<t}). $$

这就是**自回归**（autoregressive）建模——模型只能用左侧已经看到的 token 来预测当前 token。训练目标是**极大化训练语料的对数似然**，等价于最小化 next-token 预测的**交叉熵损失**：

$$ \mathcal{L} = -\sum_{t=1}^T \log p_\theta(x_t \mid x_{<t}). $$

在实现上，这等价于把 token 序列 `[x_0, x_1, ..., x_{T-1}]` 作为输入，把 `[x_1, x_2, ..., x_T]` 作为目标，让模型并行地预测每个位置的下一个 token。


## 2. 数据准备：TinyShakespeare 字符级

我们用 **TinyShakespeare** 数据集——莎士比亚作品集的纯文本（约 100 万字符）。它够小（可以塞进内存）、够有规律（押韵、对话结构），是字符级语言模型的经典测试床。

为简化起见，采用**字符级 tokenization**：每个英文字符（含标点和换行）算一个 token，词表大小约 65。


In [ ]:
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import urllib.request
from pathlib import Path

torch.manual_seed(0)


# 下载 TinyShakespeare（约 1.1 MB）
data_path = Path('tinyshakespeare.txt')
if not data_path.exists():
    url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    print(f'downloading {url}')
    urllib.request.urlretrieve(url, data_path)

text = data_path.read_text(encoding='utf-8')
print(f'corpus length (chars): {len(text):,}')
print('first 200 chars:')
print(text[:200])


In [ ]:
# 字符级词表
chars = sorted(set(text))
vocab_size = len(chars)
print(f'vocab size: {vocab_size}')
print('vocab:', ''.join(chars))

# 字符 <-> id 映射
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}


def encode(s):
    return torch.tensor([stoi[c] for c in s], dtype=torch.long)


def decode(ids):
    return ''.join(itos[int(i)] for i in ids)


# 把整个语料 encode 成一个长 Tensor
data = encode(text)
print(f'data shape: {data.shape}, dtype: {data.dtype}')

# 90% 训练 / 10% 验证
n = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]
print(f'train: {len(train_data):,}  val: {len(val_data):,}')


## 3. 构建 nanoGPT 模型

我们要实现一个**仅解码器**（decoder-only）的 Transformer，结构沿用 GPT-2/Llama 的现代设计：

```
input ids  ->  token embedding + positional embedding
               ->  N x Transformer Block
                   { LayerNorm -> Causal Multi-Head Self-Attention -> residual
                     LayerNorm -> FeedForward (4x hidden) -> residual }
               ->  final LayerNorm  ->  Linear (vocab head)
                                                       ->  logits [B, T, V]
```

几个关键点：

1. **因果掩码**：训练时模型可以"作弊地"看到未来 token，所以注意力必须用上三角 mask 屏蔽未来位置；
2. **预归一化**（pre-LN）：把 LayerNorm 放在 attention/FFN 之前——比后归一化训练更稳定；
3. **位置编码**：用可学习的位置嵌入（学起来比正余弦更灵活）。

下面是一个紧凑实现，每个组件单独成类便于理解。


In [ ]:
# ============== 因果多头自注意力 ==============
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout=0.0):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        # q, k, v 投影合在一个矩阵里，更快
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = nn.Dropout(dropout)
        # 因果掩码：上三角为 -inf
        mask = torch.triu(torch.ones(block_size, block_size), diagonal=1).bool()
        self.register_buffer('mask', mask)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)                                            # [B, T, 3*C]
        q, k, v = qkv.split(C, dim=2)                                # 3 个 [B, T, C]
        # 拆头：[B, T, n_head, head_dim] -> [B, n_head, T, head_dim]
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        # 注意力 logits：(B, n_head, T, T)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        att = att.masked_fill(self.mask[:T, :T], float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.dropout(att)
        y = att @ v                                                  # [B, n_head, T, head_dim]
        y = y.transpose(1, 2).contiguous().view(B, T, C)             # 合头
        return self.proj(y)


# ============== 前馈网络 (FFN) ==============
class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd, bias=False),
            nn.GELU(),                                                # GPT-2 风格
            nn.Linear(4 * n_embd, n_embd, bias=False),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


# ============== 单个 Transformer Block (pre-LN) ==============
class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffn = FeedForward(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


# ============== nanoGPT 整体 ==============
class NanoGPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_layer=4, n_head=4, n_embd=128, dropout=0.0):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.Sequential(*[
            Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)

        # 参数计数
        n_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f'nanoGPT params: {n_params/1e6:.2f}M')

    def forward(self, idx, targets=None):
        # idx: [B, T]
        B, T = idx.shape
        assert T <= self.block_size

        tok = self.tok_emb(idx)                                       # [B, T, C]
        pos = self.pos_emb(torch.arange(T, device=idx.device))        # [T, C]
        x = self.drop(tok + pos)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.head(x)                                         # [B, T, vocab]

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss


## 4. 预训练循环

训练流程：

1. 从语料里随机切 `batch_size` 个长度为 `block_size` 的连续片段；
2. 模型输入是片段的前 $T$ 个 token，目标是后 $T$ 个 token（错开一位）；
3. AdamW 优化器，权重衰减 0.1，学习率 warmup（500 步线性升）+ cosine decay；
4. 每 200 步在 train/val 上各采样 50 个 batch 做评估。

CPU 上跑这个小模型大约 3-5 分钟能收敛到 val loss ≈ 1.6 左右。


In [ ]:
# 超参（CPU 友好的小模型）
block_size = 64
batch_size = 32
n_layer    = 4
n_head     = 4
n_embd     = 128
dropout    = 0.1
max_iters  = 1500
eval_iters = 50
eval_every = 300
lr_max     = 3e-3
lr_min     = 1e-4
warmup     = 100

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')


def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i+block_size]       for i in ix])
    y = torch.stack([data[i+1:i+block_size+1]   for i in ix])
    return x.to(device), y.to(device)


@torch.no_grad()
def estimate_loss(model):
    """在 train/val 上随机采样 eval_iters 个 batch 求平均 loss。"""
    out = {}
    model.eval()
    for split in ('train', 'val'):
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


def get_lr(step):
    if step < warmup:
        return lr_max * step / warmup
    # cosine decay 到 lr_min
    progress = (step - warmup) / (max_iters - warmup)
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + math.cos(math.pi * progress))


# 初始化模型
torch.manual_seed(0)
model = NanoGPT(vocab_size, block_size, n_layer, n_head, n_embd, dropout).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr_max, weight_decay=0.1, betas=(0.9, 0.95))

# 训练
t0 = time.time()
for step in range(max_iters):
    # 调整学习率
    lr = get_lr(step)
    for g in optimizer.param_groups:
        g['lr'] = lr

    # 单步训练
    X, Y = get_batch('train')
    _, loss = model(X, Y)
    optimizer.zero_grad(); loss.backward(); optimizer.step()

    if step % eval_every == 0 or step == max_iters - 1:
        losses = estimate_loss(model)
        elapsed = time.time() - t0
        print(f'step {step:4d}  lr {lr:.4f}  train {losses["train"]:.4f}  val {losses["val"]:.4f}  ({elapsed:.0f}s)')
print(f'training done in {time.time()-t0:.0f}s')


## 5. 解码 / 采样策略

训练完模型后，怎么用它生成文本？给定一段前缀，模型给出下一个 token 的概率分布。我们要从这个分布**采样**一个 token，append 到序列里，再继续预测下一个，循环往复。

不同的采样策略会产生很不一样的文本：

- **Greedy**（$\arg\max$）：每次选概率最大的，输出确定但很容易陷入重复；
- **Temperature sampling**：在 logits 上除以温度 $\tau$ 后做 softmax 再采样。$\tau < 1$ 让分布更尖锐（接近 greedy），$\tau > 1$ 让分布更平坦（更随机）；
- **Top-k**：只在概率 top-$k$ 的 token 上重新归一化采样；
- **Top-p（nucleus）**：累积概率到 $p$ 为止的最小集合上采样。


In [ ]:
@torch.no_grad()
def generate(model, prompt_ids, max_new_tokens, temperature=1.0, top_k=None, top_p=None):
    """
    通用文本生成。
    - temperature: 温度
    - top_k: 只在 top-k 上重采样
    - top_p: nucleus sampling
    返回 (B, T+max_new_tokens) 的 token ids。
    """
    model.eval()
    idx = prompt_ids.clone()
    for _ in range(max_new_tokens):
        # 取最后 block_size 个 token（窗口滑动）
        idx_cond = idx if idx.size(1) <= model.block_size else idx[:, -model.block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature                    # 取最后一个时刻的 logits

        # top-k：把非 top-k 的 logits 设为 -inf
        if top_k is not None:
            v, _ = torch.topk(logits, k=top_k)
            logits[logits < v[:, [-1]]] = float('-inf')

        # top-p：按降序累积概率，砍掉超过 p 的尾部
        if top_p is not None:
            sorted_logits, sorted_idx = torch.sort(logits, descending=True)
            cum_probs = F.softmax(sorted_logits, dim=-1).cumsum(dim=-1)
            mask = cum_probs > top_p
            # 保留 cum_probs 刚刚跨过 p 的那一个 token
            mask[..., 1:] = mask[..., :-1].clone()
            mask[..., 0] = False
            sorted_logits[mask] = float('-inf')
            logits = torch.zeros_like(logits).scatter_(1, sorted_idx, sorted_logits)

        # 采样
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)
    return idx


# 不同策略生成对比
prompt = "ROMEO:"
ctx = encode(prompt).unsqueeze(0).to(device)

print('=' * 60); print('Greedy (temperature=0.01, top_k=1):')
out = generate(model, ctx, max_new_tokens=200, temperature=0.01, top_k=1)
print(decode(out[0].tolist()))

print('=' * 60); print('Temperature=1.0:')
out = generate(model, ctx, max_new_tokens=200, temperature=1.0)
print(decode(out[0].tolist()))

print('=' * 60); print('Top-k=10, Temperature=0.8:')
out = generate(model, ctx, max_new_tokens=200, temperature=0.8, top_k=10)
print(decode(out[0].tolist()))

print('=' * 60); print('Top-p=0.9, Temperature=0.8:')
out = generate(model, ctx, max_new_tokens=200, temperature=0.8, top_p=0.9)
print(decode(out[0].tolist()))


## 6. 工程优化：KV cache

上面的 `generate` 每生成一个新 token，都把整个前缀 reprocess 一遍——这是 $O(T^2)$ 的注意力计算！

但其实，每一步只有最后一个位置的 query 是新的；前面所有位置的 key 和 value **可以缓存**下来。这就是 **KV cache**：保存每一层 attention 的 key 和 value，下一步只算新位置的 q、然后和缓存的 K/V 做注意力，复杂度降到 $O(T)$。

下面是一个最小化的 KV cache 实现，把它嵌入到 `CausalSelfAttention`：


In [ ]:
class CausalSelfAttentionWithCache(nn.Module):
    """和 CausalSelfAttention 接口一致，但支持外部传入 K/V cache。"""

    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        mask = torch.triu(torch.ones(block_size, block_size), diagonal=1).bool()
        self.register_buffer('mask', mask)

    def forward(self, x, kv_cache=None):
        """
        kv_cache: 可选的 (K_prev, V_prev)，shape (B, n_head, T_prev, head_dim)
        return: y, (K_new, V_new)
        """
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        if kv_cache is not None:
            K_prev, V_prev = kv_cache
            k = torch.cat([K_prev, k], dim=2)        # 在时间维拼接
            v = torch.cat([V_prev, v], dim=2)

        T_full = k.size(2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # 因果掩码：只对当前 query 屏蔽未来位置
        # q 的位置是 [T_full - T, T_full)，可以看到 [0, T_full)
        if T > 1:                                     # 训练时 T==block_size，需要 mask
            att = att.masked_fill(self.mask[T_full-T:T_full, :T_full], float('-inf'))
        # T==1 时（增量解码），不需要 mask，因为 query 就是最后一个位置

        att = F.softmax(att, dim=-1)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y), (k, v)


# 演示：KV cache 加速 vs 不加速
print('KV cache 思想：保存每一层 attention 的 K, V，下一步增量计算。')
print('  - 无 cache：每生成 1 token 需要重算整个序列的注意力，复杂度 O(T^2)')
print('  - 有 cache：每生成 1 token 只需算与历史 K/V 的注意力，复杂度 O(T)')
print('  - 内存代价：存 K, V，约 2 * n_layer * n_head * T * head_dim 个 float')
print('  - 在 GPT/Llama 等大模型推理中，KV cache 是必备优化（10x+ 加速）')


## 小结

本节从零实现了一个字符级 nanoGPT，覆盖：

- **自回归语言建模**：next-token 预测损失，链式法则；
- **Transformer 解码器**：因果自注意力 + 前馈 + 预归一化残差结构；
- **预训练循环**：warmup + cosine decay 学习率、AdamW、cross-entropy；
- **采样策略**：greedy / temperature / top-k / top-p 的代码与对比；
- **KV cache**：增量解码工程优化的思想与实现要点。

虽然我们只训了一个 1M 参数的小模型，但所有概念和代码都和真正的 GPT-3/Llama 一模一样——只是后者参数量是 1000-1700 亿，训练数据是 1-15 万亿 token。

**真实大模型的下一步**：

- **预训练之后**：监督微调（SFT）、人类反馈对齐（RLHF/DPO）、推理增强（思维链）；
- **作为智能体**：工具调用、检索增强（RAG）、规划 + 反思（ReAct）；
- **多智能体协作**：模型之间相互沟通完成复杂任务。

这些都在下一节"大语言模型与智能体（下）"中演示。
